In [6]:
!pip uninstall mediapipe opencv-python opencv-python-headless

Found existing installation: mediapipe 0.10.11
Uninstalling mediapipe-0.10.11:
  Would remove:
    /home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/mediapipe-0.10.11.dist-info/*
    /home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/mediapipe.libs/libEGL-29b68110.so.1.1.0
    /home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/mediapipe.libs/libGLESv2-3a91e100.so.2.1.0
    /home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/mediapipe.libs/libGLdispatch-dcc1ca97.so.0.0.0
    /home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/mediapipe/*
Proceed (y/n)? ^C
ERROR: Operation cancelled by user


In [9]:
!pip show opencv-python

Name: opencv-python
Version: 4.10.0.84
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: None
Author-email: None
License: Apache 2.0
Location: /home/mojtaba.kananisarche/.local/lib/python3.8/site-packages
Requires: numpy, numpy
Required-by: 


In [1]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import random

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Root directory containing all videos
merged_folder = 'Raw'

# Directory to save extracted landmarks
landmark_dir = "face_landmarks"
os.makedirs(landmark_dir, exist_ok=True)

# Directory to save images with landmarks
landmark_images_dir = "landmark_images"
os.makedirs(landmark_images_dir, exist_ok=True)

index = 0
# Process all videos in the merged folder
for root, _, files in os.walk(merged_folder):
    for file_name in files:
        print(f"i = {index}")
        index += 1
        
        if file_name.endswith(('.mp4', '.mov', '.MOV')):  # Process only video files
            if file_name[:-3]+"csv" in sum([os.listdir(f"face_landmarks/F{i+1}/Raw") for i in range(5)],[]) or file_name.startswith('1'):
                print(f"Skipping file {file_name}")
                continue
            print(f"starting to work on {file_name}")
            video_path = os.path.join(root, file_name)
            video_name, video_extension = os.path.splitext(file_name)
            landmarks_data = []

            # Open the video
            cap = cv2.VideoCapture(video_path)

            frame_index = 0
            random_frame_index = random.randint(0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1)
            saved_image = False

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                # Convert the frame to RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Process face landmarks
                face_results = face_mesh.process(frame_rgb)

                if face_results.multi_face_landmarks:
                    for face_landmarks in face_results.multi_face_landmarks:
                        # Extract x, y coordinates for all 468 landmarks
                        for i, landmark in enumerate(face_landmarks.landmark):
                            landmarks_data.append({
                                "video": video_name,
                                "frame": frame_index,
                                "landmark_index": i,
                                "x": landmark.x,
                                "y": landmark.y
                            })

                        # Save an image with landmarks on the random frame
                        if frame_index == random_frame_index and not saved_image:
                            for landmark in face_landmarks.landmark:
                                x = int(landmark.x * frame.shape[1])
                                y = int(landmark.y * frame.shape[0])
                                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)
                            output_image_path = os.path.join(landmark_images_dir, f"{video_name}.jpg")
                            cv2.imwrite(output_image_path, frame)
                            print(f"Landmark image saved for {video_name} at {output_image_path}")
                            saved_image = True

                frame_index += 1

            # Save the landmarks to a CSV file
            landmarks_df = pd.DataFrame(landmarks_data)
            output_csv_path = os.path.join(landmark_dir, f"{video_name}.csv")
            landmarks_df.to_csv(output_csv_path, index=False)
            print(f"Landmarks saved for {video_name} in {output_csv_path}")

            cap.release()

# Close the Face Mesh model
face_mesh.close()


KeyboardInterrupt: 